In [0]:
from typing import (
    Callable,
    Protocol,
    runtime_checkable
)

from pyspark.sql.functions import (
    col, explode, lit,
    current_timestamp, from_unixtime, to_timestamp,
    DataFrame, initcap, 
    date_format, lpad, floor, concat,
)
import pandas as pd
import numpy as np
from pydantic import (
    BaseModel,
    field_validator,
)

In [0]:
import os
import sys
sys.path.append("/Repos/stan.mng@gmail.com/fflogs_graphql/jobs")

from utils import (
    roundfloat,
    to_title_case,
    trim_whitespace,
    epoch_to_timestamp
)
from settings import fflogs_settings as settings

## Bronze: Ingest as is

In [0]:
def preview_bronze_sql() -> None:
    query = f"""
    SELECT 
        * EXCEPT(_rescued_data),
        CURRENT_TIMESTAMP() AS ldts,
        '{settings.SECRET_SCOPE}' AS rsrc
    FROM READ_FILES (
        '{settings.volume_dir}/sample/*.json',
        format => 'json',
        multiLine => 'true'
    )
    LIMIT 5
    """
    df = spark.sql(query)

    display(df)

In [0]:
class FFLogsEncounterRankingBronze:
    def __init__(self, volume_dir: str, source: str) -> None:
        self.df = (spark.read
            .json(f"{volume_dir}/*.json", multiLine=True)
            .withColumn("ldts", current_timestamp())
            .withColumn("rsrc", lit(source))
        )

In [0]:
@runtime_checkable
class SilverTable(Protocol):
    df: DataFrame

    def unpack(self) -> "SilverTable": ...
    def define(self) -> "SilverTable": ...
    def clean(self) -> "SilverTable": ...

In [0]:
class FFLogsEncounterRankingSilver(SilverTable):
    def __init__(self, bronze: "BronzeTable") -> None:
        self.df = bronze.df

    def unpack(self) -> "FFLogsEncounterRankingSilver":
        self.df = self.df.select(
            # ===== Metadata
            "ldts",
            "rsrc",
            # ===== Encounters
            col("worldData.encounter.id").alias("encounter_id"),
            col("worldData.encounter.name").alias("encounter_name"),
            # ===== Record Unpacking
            explode("worldData.encounter.characterRankings.rankings").alias("r")
        )
        return self

    def define(self) -> "FFLogsEncounterRankingSilver":
        self.df = self.df.select(
            # ===== Metadata
            col("ldts"),
            col("rsrc"),
            # ===== Encounters
            col("encounter_id"),
            col("encounter_name"),
            # ===== Ranking Record
            col("r.name").alias("player_name"),
            col("r.spec").alias("class"),
            col("r.amount").alias("rDPS"),
            col("r.aDPS").alias("aDPS"),
            col("r.startTime").alias("upload_epoch"),
            col("r.duration").alias("duration"),
            # ===== Normalizable Foreign Entities
            col("r.lodestoneID").alias("lodestoneID"),
            col("r.guild.id").alias("guild_id"),
            col("r.guild.name").alias("guild_name"),
            col("r.server.id").alias("server_id"),
            col("r.server.name").alias("server_name"),
        )
        return self

    def clean(self) -> "FFLogsEncounterRankingSilver":
        self.df = self.df.select(
            # ===== Metadata
            col("ldts"),
            col("rsrc"),
            # ===== Encounters
            col("encounter_id"),
            trim_whitespace(initcap(col("encounter_name")))
                .alias("encounter_name"),
            # ===== Ranking Record
            trim_whitespace(initcap(col("player_name")))
                .alias("player_name"),
            to_title_case(trim_whitespace(col("class")), "pascal")
                .alias("class"),
            roundfloat(col("rDPS"), 2).alias("rDPS"),
            roundfloat(col("aDPS"), 2).alias("aDPS"),
            epoch_to_timestamp(col("upload_epoch"), "ms")
                .alias("ranked_date"),
            (col("duration") / 1000).cast("int").alias("duration"),
            # ===== Normalizable Foreign Entities
            col("lodestoneID").cast("bigint").alias("lodestoneID"),
            col("guild_id").cast("bigint").alias("guildID"),
            trim_whitespace(col("guild_name")).alias("guild_name"),
            col("server_id").cast("int").alias("server_id"),
            trim_whitespace(initcap(col("server_name")))
                .alias("server_name"),
        )

        return self


In [0]:
FFLogsEncounterRankingSilver(
    BronzeTable(
        volume_dir=settings.volume_dir,
        source=settings.SECRET_SCOPE
    )
).unpack().define().clean().df.show()

In [0]:
class FFLogsEncounterRankingGold:

    def __init__(self, silver: SilverTable):
        self.df = silver.df.select(
            col("encounter_name"),
            col("player_name"),
            col("class"),
            col("rDPS"),
            col("aDPS"),
            date_format(col("ranked_date"), "yyyy-MM-dd").alias("ranked_date"),
            col("duration").alias("duration"),  # duration in seconds
            lpad(floor(col("duration") / 60).cast("string"), 2, "0").alias("MM"),
            lpad((col("duration") % 60).cast("string"), 2, "0").alias("SS"),
            col("lodestoneID"),
            col("guild_name"),
            col("server_name")
        ).withColumn(
            "duration_fmt",
            concat(col("MM"), lit(":"), col("SS"))
        ).select(
            "encounter_name",
            "player_name",
            "class",
            "rDPS",
            "aDPS",
            "ranked_date",
            "duration_fmt",
            "duration",
            "lodestoneID",
            "guild_name",
            "server_name"
        )

In [0]:
def draft_pipeline():
    bronze_table = BronzeTable(
        volume_dir=settings.volume_dir,
        source=settings.SECRET_SCOPE
    )
    silver_table = (FFLogsEncounterRankingSilver(bronze_table)
        .unpack()
        .define()
        .clean()
    )
    gold_table = FFLogsEncounterRankingGold(silver_table)
    return gold_table

In [0]:
draft_pipeline().df.show()